In [ ]:
# %pip install pandas requests

import os
import time
import urllib.parse as up
import pandas as pd
import requests
from pathlib import Path

# === Config ===
API_KEY = "YOUR-TINY-URL-API-KEY-HERE"
RATE_DELAY_SEC = 0.5

# --- Paths ---
desktop = Path.home() / "Desktop" 
input_csv = desktop / "PATH/TO/RERUN/LINKS"
output_csv = desktop / "PATH/FOR/LINKS/TO/GO"

# --- Load CSV ---
df_raw = pd.read_csv(input_csv, header=None, names=["url"])
urls = df_raw.iloc[:, 0].astype(str).str.strip()

def make_custom_code(url: str) -> str:
    path = up.urlparse(url).path
    last = Path(path).name
    base = last.replace(".html", "").strip().lower()
    # Remove underscores/dashes and use new suffix style
    base = base.replace("_", "").replace("-", "")
    return f"{base}ldgrp2025"

def _stringify_tinyurl_errors(err_val) -> str:
    if err_val is None:
        return ""
    if isinstance(err_val, str):
        return err_val
    if isinstance(err_val, list):
        return "; ".join(str(e.get("message", e)) if isinstance(e, dict) else str(e) for e in err_val)
    if isinstance(err_val, dict):
        return err_val.get("message", "") or str(err_val)
    return str(err_val)

def shorten_tinyurl(long_url: str, custom_code: str, timeout=20):
    endpoint = "https://api.tinyurl.com/create"
    payload = {"url": long_url, "domain": "tinyurl.com", "alias": custom_code}
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"

    try:
        r = requests.post(endpoint, json=payload, headers=headers, timeout=timeout)
    except requests.RequestException as e:
        return "", f"network_error: {e}"

    try:
        data = r.json()
    except Exception:
        return "", f"parse_error: HTTP {r.status_code} body={r.text[:200]}"

    if r.status_code == 200 and "data" in data and "tiny_url" in data["data"]:
        return data["data"]["tiny_url"], "ok"

    msg = _stringify_tinyurl_errors(data.get("errors")) or data.get("message", f"HTTP {r.status_code}")
    if r.status_code in (401, 403):
        msg += " (auth issue: check API key/token & plan allows custom alias)"
    return "", f"tinyurl_error: {msg}"

rows = []
for url in urls:
    if not url or not url.startswith(("http://", "https://")):
        rows.append({
            "original_url": url,
            "custom_code": "",
            "short_url": "",
            "status": "skip",
            "api_message": "invalid_or_missing_url",
        })
        continue

    code = make_custom_code(url)
    short_url, msg = shorten_tinyurl(url, code)
    rows.append({
        "original_url": url,
        "custom_code": code,
        "short_url": short_url,
        "status": "created" if short_url else "failed",
        "api_message": msg,
    })
    time.sleep(RATE_DELAY_SEC)

out_df = pd.DataFrame(rows, columns=["original_url","custom_code","short_url","status","api_message"])
out_df.to_csv(output_csv, index=False)
print(out_df["status"].value_counts(dropna=False).to_string())
print(f"\nSaved -> {output_csv}")
